# Part B — Retrieval Failure Cases & Fixes
**CS4241 | Student: Farima Konaré | Index: 10012200004**

This notebook demonstrates failure cases in retrieval and the fix applied.

**Failure case:** Vague/ambiguous queries return irrelevant chunks.
**Fix:** Query expansion — prepend domain-specific context keywords before embedding.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

from retrieval.embedder import get_embedder
from retrieval.vector_store import VectorStore
from retrieval.retriever import HybridRetriever
from ingestion.chunker import load_chunks

# Load index
chunks = load_chunks('all_chunks.json')
emb = get_embedder()
store = VectorStore(dim=emb.dim)
if not store.load():
    store.build(chunks, emb)
    store.save()

retriever = HybridRetriever(store, chunks)
print(f'Index loaded: {store.size} vectors')

## Failure Case 1: Vague geographic query

In [ ]:
FAILURE_QUERY = 'What happened in the north?'

q_vec = emb.encode_query(FAILURE_QUERY)
results = retriever.retrieve(FAILURE_QUERY, q_vec, k=3)

print(f'Query: "{FAILURE_QUERY}"')
print('\n--- BEFORE fix (no query expansion) ---')
for i, r in enumerate(results, 1):
    print(f'{i}. combined={r["combined_score"]:.3f} | source={r["chunk"]["metadata"]["source"]}')
    print(f'   {r["chunk"]["text"][:120]}')

In [ ]:
# Apply fix: query expansion
expanded = retriever.expand_query(FAILURE_QUERY, source_filter='election')
print(f'Expanded query: "{expanded}"')

q_vec_expanded = emb.encode_query(expanded)
results_fixed = retriever.retrieve(expanded, q_vec_expanded, k=3)

print('\n--- AFTER fix (with query expansion) ---')
for i, r in enumerate(results_fixed, 1):
    print(f'{i}. combined={r["combined_score"]:.3f} | source={r["chunk"]["metadata"]["source"]}')
    print(f'   {r["chunk"]["text"][:120]}')

## Failure Case 2: Out-of-domain query (no relevant information)

In [ ]:
OOD_QUERY = 'What is the capital of France?'

q_vec_ood = emb.encode_query(OOD_QUERY)
results_ood = retriever.retrieve(OOD_QUERY, q_vec_ood, k=3)

print(f'Query: "{OOD_QUERY}"')
print(f'Is relevant (threshold ≥ 0.25): {retriever.is_relevant(results_ood)}')
print()
for i, r in enumerate(results_ood, 1):
    print(f'{i}. combined={r["combined_score"]:.3f} | {r["chunk"]["text"][:80]}')

## Summary

| Failure | Root cause | Fix | Result |
|---|---|---|---|
| Vague northern query | "north" matches everything; no domain anchor | Query expansion: prepend "Ghana election northern region" | Top results now come from northern regional election data |
| Out-of-domain query | No relevant chunks exist | Confidence threshold (< 0.25) | Pipeline returns "no information" message instead of hallucinating |